# Extreme periods

Clustering finds *typical* behavior, so a once-a-year peak gets blended into an average and lost.
`ExtremeConfig` forces chosen extremes to be kept exactly, alongside the normal clustering.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## The peak gets averaged away

In [ ]:
base = tsam.aggregate(data, n_clusters=6, period_duration="1D")
print(f"original peak Load:  {data['Load'].max():.1f}")
print(f"typical-period peak: {base.cluster_representatives['Load'].max():.1f}")

## Keep it exactly

Target the period holding the single highest/lowest value (`max_value`/`min_value`) or the
highest/lowest period total (`max_period`/`min_period`):

In [ ]:
from tsam import ExtremeConfig

kept = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"]),
)
print(f"clusters: {base.n_clusters} -> {kept.n_clusters}")
print(f"typical-period peak now: {kept.cluster_representatives['Load'].max():.1f}")

## How the extreme enters: `method`

- **`new_cluster`** — add it as its own cluster, kept exactly, never blended. Recommended.
- **`append`** — add it as an extra typical period.
- **`replace`** — substitute the most similar existing cluster (cluster count unchanged).

In [ ]:
for method in ["new_cluster", "append", "replace"]:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        extremes=ExtremeConfig(method=method, max_value=["Load"]),
    )
    print(
        f"{method:12s} -> {r.n_clusters} clusters, "
        f"peak {r.cluster_representatives['Load'].max():.1f}"
    )

## Several at once

Request as many as you need. A period that is extreme on more than one criterion is added once.

In [ ]:
multi = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"], min_value=["T"]),
)
print(f"clusters: {base.n_clusters} -> {multi.n_clusters}")
kept.plot.cluster_counts()

## Keeping the cluster count fixed

`append` and `new_cluster` add the extremes *on top* of `n_clusters`, and how many they add
depends on the data: two criteria can resolve to the same period, or an extreme can coincide
with a cluster the run produced anyway. Both collapse, so the same configuration can yield a
different number of typical periods on a different dataset — awkward when results from several
runs have to line up.

`preserve_n_clusters=True` carves the extremes out of the budget instead. The distinct extremes
are detected first, the remaining periods are clustered into `n_clusters - D` groups, and the
`D` extremes are added back — exactly `n_clusters` typical periods, whatever the data. The
trade is fewer regular clusters, so ordinary periods are represented a little more coarsely. It
needs `n_clusters > D`, and it is a no-op for `replace`, which never changes the count.

In [ ]:
fixed = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(
        method="new_cluster",
        max_value=["Load"],
        min_value=["T"],
        preserve_n_clusters=True,
    ),
)
print(
    f"without: {multi.n_clusters} clusters, with preserve_n_clusters: {fixed.n_clusters}"
)

Extreme days stand in for just themselves, so they carry a small occurrence count next to the
typical clusters. They are also **excluded from rescaling** by design, so unlike a `maxoid`
representation they do not fight `preserve_column_means`.

---

* [Choosing a method](../tutorials/choosing_a_method.ipynb) — the three different ways to preserve
  a peak, and why they are not interchangeable.
* [Representations](representations.ipynb) — `maxoid` / `minmax_mean`, the lighter touch.
* [Rescaling](../explanation/how-aggregation-works/05_rescaling.ipynb) — how mean preservation
  works.